In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.interpolate import interp1d


In [ ]:
# ── 1. Simulate raw pupil data ──────────────────────
np.random.seed(42)  # create a syn data of pupil
time = np.linspace(0, 10, 900)  # 10 seconds at 90Hz

In [ ]:
true_signal = 3.0 + 0.3 * np.sin(2 * np.pi * 0.2 * time) #3.0mm is avg pupil size , 0.3 mmm is pupil chnage

In [ ]:
# Add noise
noise = 0.05 * np.random.randn(900)

In [ ]:
raw = true_signal + noise # mix real signal with noise

In [ ]:
raw[200:210] = 0   # blink 1
raw[500:515] = 0   # blink 2
raw[750:760] = 0   # blink 3

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(time, raw, alpha=0.6, color='steelblue', label='Raw signal')
plt.title('Raw Pupil Diameter Signal (with blinks)')
plt.xlabel('Time (s)')
plt.ylabel('Pupil Diameter (mm)')
plt.legend()
plt.show()

In [ ]:
cleaned = raw.copy().astype(float)  # to store NAN values

In [ ]:
cleaned[cleaned < 1.5] = np.nan

In [ ]:
valid = ~np.isnan(cleaned)

In [ ]:
interp_func = interp1d(time[valid], cleaned[valid], kind='linear', fill_value='extrapolate')

In [ ]:
interpolated = interp_func(time)

In [ ]:
b, a = signal.butter(3, 0.1, btype='low')

In [ ]:
filtered = signal.filtfilt(b, a, interpolated)

In [ ]:
baseline = np.mean(filtered[:45])

In [ ]:
corrected = filtered - baseline

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(time, raw, alpha=0.3, color='gray', label='Raw (with blinks)')
plt.plot(time, interpolated, alpha=0.5, color='orange', label='Interpolated')
plt.plot(time, filtered, color='steelblue', linewidth=2, label='Filtered')
plt.plot(time, corrected, color='green', linewidth=2, label='Baseline corrected')

In [ ]:
print(f"Mean:   {np.mean(corrected):.4f}")

In [ ]:
print(f"Std:    {np.std(corrected):.4f}")

In [ ]:
print(f"MAD:    {np.median(np.abs(corrected - np.median(corrected))):.4f}")

In [ ]:
print(f"Min:    {np.min(corrected):.4f}")
print(f"Max:    {np.max(corrected):.4f}")
print(f"Range:  {np.max(corrected) - np.min(corrected):.4f}")